In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

In [6]:
pyPdfLoader = PyPDFLoader("../data/Sri_krishna_medical_report_medibuddy.pdf")
docs = pyPdfLoader.load()
len(docs)

13

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = splitter.split_documents(docs)
len(split_docs)

28

In [8]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectordb = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    collection_name="medical_reports",
)

In [12]:
@tool
def retriever_tool(query: str):
    """
        This tool can help you to retrieve the relevant data of the PDF Documents, and these pdf
        documents have details about medical reports.
    """
    
    print("Tool called:", query)
    results = vectordb.similarity_search(query=query, k=4)
    context = ""
    for doc in results:
        context += doc.page_content + "\n"
        
    return context

In [13]:
llm = ChatGroq(model = "openai/gpt-oss-20b")

In [21]:
system_prompt = """
    You are a helpful assistant that answers questions using retrieved context.
	ALWAYS use the `retriever_tool` tool for questions requiring external knowledge.
"""

In [22]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=system_prompt
)

In [27]:
query = "What is the name of the patient and what are the parameters that are lower or higher than normal?"
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": query
        }
    ]
})
print(response["messages"][-1].content)

Tool called: patient name abnormal parameters lower higher normal medical report
**Patient name:**  
**MR. SRI KRISHNA KOLLIPARA**

**Parameters outside the normal reference ranges**

| Test | Result | Reference range | Status |
|------|--------|-----------------|--------|
| LDL Cholesterol | 109 mg/dL | < 100 mg/dL (normal) – 100–129 mg/dL (desirable) | **Higher than normal** (above the normal cut‑off) |

All other measured parameters (fasting glucose, total cholesterol, HDL cholesterol, sodium, potassium, chloride, and TSH) fall within their respective normal reference ranges. No values are lower than normal.
